> **Versión corregida (v2.13, línea v1 de despliegue, ejecución verificada)** — generada a partir de `notebooks/09_evaluacion_conjunta_modelos_corregido.ipynb` sin sobrescribirlo.
>
> Cambios aplicados:
> - Celda de inicialización: se sustituyen la detección/montaje de Drive y el clonado de `main` por el bootstrap portable (`PROJECT_ROOT`/`REPO_ROOT`, rama `feature/entorno-capacidad-v2`).
> - Renombrado semántico: el antiguo `PROJECT_ROOT` (copia del repositorio) pasa a `REPO_ROOT`; `PERSISTENT_ROOT` se conserva como raíz de la línea v1 (= `PROJECT_ROOT`, o `PROJECT_ROOT/prueba_linea_v1` en modo prueba).
>
> Rutas: `PROJECT_ROOT` = datos/modelos/resultados (Drive en Colab, o `PIDA_PROJECT_ROOT`); `REPO_ROOT` = código. Modo prueba: `PIDA_MODO_PRUEBA=1` (cohorte pequeña, pocos pasos, salida en `prueba_linea_v1/`).


# PIDA-RL Diabetes — 09. Evaluación conjunta de modelos

Consolida las evaluaciones de políticas de referencia, DQN, Q-learning y PPO. Este notebook no reentrena modelos ni altera la cohorte.

> Las métricas proceden de un simulador y permiten comparar políticas bajo supuestos comunes; no prueban efectividad clínica real.


## 1. Objetivo

Generar una tabla canónica de resultados por política, semilla y nivel de riesgo; calcular indicadores agregados, cobertura de alto riesgo, cumplimiento de capacidad y un ranking técnico reproducible.


## 2. Inicialización


In [ ]:
# === Bootstrap portable: PROJECT_ROOT (corrección v2.12) ===
# PROJECT_ROOT = raíz persistente de datos, modelos y resultados.
# REPO_ROOT    = copia local del código (repositorio GitHub).
# Prioridad: variables de entorno PIDA_PROJECT_ROOT / PIDA_REPO_ROOT,
# después Google Colab + Drive y, por último, ejecución local.
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import importlib
import importlib.util
import subprocess

PROJECT_NAME = 'pida-rl-datos-publicos'

REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
BRANCH = os.environ.get('PIDA_BRANCH', 'main')
PROJECT_FOLDER_NAME = 'PIDA-RL-Diabetes'
IN_COLAB = 'google.colab' in sys.modules

DRIVE_ROOT = None
if os.environ.get('PIDA_PROJECT_ROOT'):
    PROJECT_ROOT = Path(os.environ['PIDA_PROJECT_ROOT']).expanduser().resolve()
elif IN_COLAB:
    DRIVE_ROOT = next(
        (p for p in [Path('/content/drive'), Path('/content/gdrive')] if (p / 'MyDrive').is_dir()),
        None,
    )
    if DRIVE_ROOT is None:
        from google.colab import drive
        mount_point = Path('/content/drive')
        if mount_point.exists() and any(mount_point.iterdir()):
            mount_point = Path('/content/gdrive')
        drive.mount(str(mount_point), force_remount=False)
        DRIVE_ROOT = mount_point
    PROJECT_ROOT = DRIVE_ROOT / 'MyDrive' / PROJECT_FOLDER_NAME
else:
    PROJECT_ROOT = (Path.cwd() / PROJECT_FOLDER_NAME).resolve()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


def _buscar_repo(inicio):
    for candidato in [inicio, *inicio.parents]:
        if (candidato / 'src' / 'environment' / 'diabetes_followup_env.py').is_file():
            return candidato
    return None


if os.environ.get('PIDA_REPO_ROOT'):
    REPO_ROOT = Path(os.environ['PIDA_REPO_ROOT']).expanduser().resolve()
else:
    REPO_ROOT = _buscar_repo(Path.cwd().resolve()) or (
        Path('/content/pida-rl-datos-publicos') if IN_COLAB
        else Path.cwd().resolve() / 'pida-rl-datos-publicos'
    )
if not REPO_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])
elif not (REPO_ROOT / 'src').is_dir():
    raise RuntimeError(f'REPO_ROOT no contiene la carpeta src: {REPO_ROOT}')
elif IN_COLAB and (REPO_ROOT / '.git').is_dir():
    # En Colab se alinea la copia temporal con la rama de trabajo (sin push ni commit).
    for args in (['fetch', 'origin', BRANCH], ['checkout', BRANCH], ['pull', '--ff-only', 'origin', BRANCH]):
        subprocess.run(['git', '-C', str(REPO_ROOT), *args], check=False)

SRC_ROOT = REPO_ROOT / 'src'
for ruta in (str(REPO_ROOT), str(SRC_ROOT)):
    if ruta not in sys.path:
        sys.path.insert(0, ruta)
os.chdir(REPO_ROOT)

# Estructura persistente común (antes: rutas fijas /content/drive/MyDrive/PIDA-RL-Diabetes).
PERSISTENT_ROOT = PROJECT_ROOT  # alias de compatibilidad con los notebooks originales
MYDRIVE_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_ROOT / 'raw'
INTERIM_DATA_DIR = DATA_ROOT / 'interim'
PROCESSED_DATA_DIR = DATA_ROOT / 'processed'
METADATA_DIR = DATA_ROOT / 'metadata'
SPECIFICATIONS_DIR = DATA_ROOT / 'specifications'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
COHORT_PATH = PROCESSED_DATA_DIR / 'cohorte_diabetes.csv'

# Modo prueba: cohorte pequeña y pocos episodios (PIDA_MODO_PRUEBA=1).
MODO_PRUEBA = os.environ.get('PIDA_MODO_PRUEBA', '0') == '1'
N_COHORTE_PRUEBA = int(os.environ.get('PIDA_N_COHORTE', '120'))
SEMILLA_MUESTRA_PRUEBA = 2026
RESULTS_V2_DIR = RESULTS_DIR / ('capacidad_v2_prueba' if MODO_PRUEBA else 'capacidad_v2')


def muestra_cohorte_prueba(cohorte, n=None, semilla=SEMILLA_MUESTRA_PRUEBA):
    """Devuelve (cohorte, factor). En modo prueba toma una submuestra
    estratificada por categoria_riesgo (bajo/medio/alto); no crea columnas."""
    n = N_COHORTE_PRUEBA if n is None else int(n)
    if not MODO_PRUEBA or len(cohorte) <= n:
        return cohorte, 1.0
    fraccion = n / len(cohorte)
    muestra = (
        cohorte.groupby('categoria_riesgo', group_keys=False)
        .sample(frac=fraccion, random_state=semilla)
        .sort_values('FOLIO_INT')
        .reset_index(drop=True)
    )
    return muestra, len(muestra) / len(cohorte)


def escalar_capacidad(capacidad, factor):
    """Mantiene la presión de capacidad (recursos/persona) al reducir la cohorte."""
    return capacidad if factor >= 1.0 else max(1, int(round(capacidad * factor)))

print('Entorno:', 'Google Colab' if IN_COLAB else 'local')
print('PROJECT_ROOT (datos/resultados):', PROJECT_ROOT)
print('REPO_ROOT (código):', REPO_ROOT)
print('Modo prueba:', MODO_PRUEBA)

# --- Línea v1 de despliegue (entorno autocontenido) ---
# En modo prueba se escribe en una carpeta aparte para no sobrescribir resultados completos ni el dashboard publicado.
LINEA_V1_ROOT = PROJECT_ROOT / 'prueba_linea_v1' if MODO_PRUEBA else PROJECT_ROOT
PERSISTENT_ROOT = LINEA_V1_ROOT
COHORTE_CANONICA = PROJECT_ROOT / 'data' / 'processed' / 'cohorte_diabetes.csv'
print('Raíz de la línea v1 (PERSISTENT_ROOT):', PERSISTENT_ROOT)

# --- Celda original (sin montaje de Drive ni clonado fijo) ---
from pathlib import Path
import os
import subprocess
import json
from datetime import datetime, timezone

TABLES_DIR = PERSISTENT_ROOT / 'results' / 'tables'
FIGURES_DIR = PERSISTENT_ROOT / 'results' / 'figures'
METADATA_DIR = PERSISTENT_ROOT / 'data' / 'metadata'

for directory in [TABLES_DIR, FIGURES_DIR, METADATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

os.chdir(REPO_ROOT)
print('Proyecto temporal:', REPO_ROOT)
print('Tablas:', TABLES_DIR)
print('Figuras:', FIGURES_DIR)


## 3. Archivos de evaluación


In [ ]:
INPUT_FILES = {
    'politicas_base': TABLES_DIR / 'evaluacion_politicas_base_v1.csv',
    'dqn_v1': TABLES_DIR / 'evaluacion_dqn_v1.csv',
    'q_learning_v1': TABLES_DIR / 'evaluacion_q_learning_v1.csv',
    'ppo_v1': TABLES_DIR / 'evaluacion_ppo_v1.csv',
}

for name, path in INPUT_FILES.items():
    print(name, '| existe:', path.is_file(), '| ruta:', path)

missing_files = [str(path) for path in INPUT_FILES.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        'Faltan archivos de evaluación. Ejecuta los notebooks correspondientes antes de continuar:\n'
        + '\n'.join(missing_files)
    )


## 4. Consolidar evaluaciones


In [ ]:
import numpy as np
import pandas as pd

required_columns = [
    'politica', 'semilla', 'nivel_riesgo', 'categoria_riesgo',
    'decisiones', 'contactos_aceptados', 'eventos_adversos',
    'recursos_usados', 'excesos_capacidad', 'cobertura_porcentaje',
    'recompensa_acumulada_episodio', 'pasos_episodio',
]

frames = []
for source_name, path in INPUT_FILES.items():
    frame = pd.read_csv(path)
    missing_columns = [column for column in required_columns if column not in frame.columns]
    if missing_columns:
        raise KeyError(
            f'El archivo {path.name} no tiene las columnas requeridas: ' + ', '.join(missing_columns)
        )
    frame = frame[required_columns].copy()
    frame['archivo_origen'] = path.name
    frame['fuente_evaluacion'] = source_name
    frames.append(frame)

df_evaluacion_conjunta = pd.concat(frames, ignore_index=True)

expected_policies = {
    'aleatoria', 'reglas_riesgo', 'reglas_capacidad',
    'dqn_v1', 'q_learning_v1', 'ppo_v1',
}
observed_policies = set(df_evaluacion_conjunta['politica'].unique())
missing_policies = expected_policies - observed_policies
if missing_policies:
    raise ValueError('No se encontraron todas las políticas esperadas: ' + ', '.join(sorted(missing_policies)))

numeric_columns = [
    'decisiones', 'contactos_aceptados', 'eventos_adversos', 'recursos_usados',
    'excesos_capacidad', 'cobertura_porcentaje', 'recompensa_acumulada_episodio',
    'pasos_episodio',
]
assert df_evaluacion_conjunta['semilla'].isin([42, 123, 2026]).all()
assert df_evaluacion_conjunta['categoria_riesgo'].isin(['bajo', 'medio', 'alto']).all()
assert np.isfinite(df_evaluacion_conjunta[numeric_columns].to_numpy(dtype=float)).all()

print('Filas consolidadas:', len(df_evaluacion_conjunta))
print('Políticas:', sorted(observed_policies))
display(df_evaluacion_conjunta.sort_values(['politica', 'semilla', 'nivel_riesgo']))


## 5. Validación de protocolo


In [ ]:
protocol_check = (
    df_evaluacion_conjunta
    .groupby('politica', as_index=False)
    .agg(
        semillas=('semilla', 'nunique'),
        niveles_riesgo=('categoria_riesgo', 'nunique'),
        pasos_minimos=('pasos_episodio', 'min'),
        pasos_maximos=('pasos_episodio', 'max'),
        decisiones_minimas=('decisiones', 'min'),
    )
)

display(protocol_check)
assert (protocol_check['semillas'] == 3).all(), 'No todas las políticas tienen tres semillas.'
assert (protocol_check['niveles_riesgo'] == 3).all(), 'No todas las políticas tienen tres riesgos.'
assert (protocol_check['pasos_minimos'] == protocol_check['pasos_maximos']).all(), 'Hay episodios con diferente número de pasos dentro de una política.'

expected_steps = int(df_evaluacion_conjunta['pasos_episodio'].mode().iloc[0])
assert (df_evaluacion_conjunta['pasos_episodio'] == expected_steps).all(), 'Las políticas no comparten el mismo horizonte de evaluación.'
print('Protocolo consistente. Pasos por episodio:', expected_steps)


## 6. Indicadores agregados


In [ ]:
episode_level = (
    df_evaluacion_conjunta
    .groupby(['politica', 'semilla'], as_index=False)
    .agg(
        recompensa_acumulada=('recompensa_acumulada_episodio', 'first'),
        pasos=('pasos_episodio', 'first'),
        decisiones=('decisiones', 'sum'),
        contactos_aceptados=('contactos_aceptados', 'sum'),
        eventos_adversos=('eventos_adversos', 'sum'),
        recursos_usados=('recursos_usados', 'sum'),
        excesos_capacidad=('excesos_capacidad', 'sum'),
    )
)

episode_level['cobertura_global_porcentaje'] = (
    100.0 * episode_level['contactos_aceptados'] / episode_level['decisiones']
)

df_resumen_modelos = (
    episode_level
    .groupby('politica', as_index=False)
    .agg(
        recompensa_media=('recompensa_acumulada', 'mean'),
        recompensa_desviacion=('recompensa_acumulada', 'std'),
        cobertura_global_media=('cobertura_global_porcentaje', 'mean'),
        cobertura_global_desviacion=('cobertura_global_porcentaje', 'std'),
        contactos_medios=('contactos_aceptados', 'mean'),
        eventos_adversos_medios=('eventos_adversos', 'mean'),
        recursos_usados_medios=('recursos_usados', 'mean'),
        excesos_capacidad_medios=('excesos_capacidad', 'mean'),
        pasos_por_episodio=('pasos', 'first'),
    )
)

df_resumen_modelos = df_resumen_modelos.round(3)
display(df_resumen_modelos.sort_values('recompensa_media', ascending=False))


## 7. Cobertura por nivel de riesgo


In [ ]:
df_cobertura_riesgo = (
    df_evaluacion_conjunta
    .groupby(['politica', 'nivel_riesgo', 'categoria_riesgo'], as_index=False)
    .agg(
        cobertura_media=('cobertura_porcentaje', 'mean'),
        cobertura_desviacion=('cobertura_porcentaje', 'std'),
        contactos_medios=('contactos_aceptados', 'mean'),
        decisiones_medias=('decisiones', 'mean'),
        eventos_adversos_medios=('eventos_adversos', 'mean'),
        recursos_usados_medios=('recursos_usados', 'mean'),
        excesos_capacidad_medios=('excesos_capacidad', 'mean'),
    )
)

df_cobertura_riesgo = df_cobertura_riesgo.round(3)
display(df_cobertura_riesgo.sort_values(['politica', 'nivel_riesgo']))

high_risk_coverage = (
    df_cobertura_riesgo
    .query('categoria_riesgo == "alto"')
    [['politica', 'cobertura_media', 'cobertura_desviacion']]
    .rename(columns={
        'cobertura_media': 'cobertura_alto_riesgo_media',
        'cobertura_desviacion': 'cobertura_alto_riesgo_desviacion',
    })
)

medium_risk_coverage = (
    df_cobertura_riesgo
    .query('categoria_riesgo == "medio"')
    [['politica', 'cobertura_media']]
    .rename(columns={'cobertura_media': 'cobertura_medio_riesgo_media'})
)

low_risk_coverage = (
    df_cobertura_riesgo
    .query('categoria_riesgo == "bajo"')
    [['politica', 'cobertura_media']]
    .rename(columns={'cobertura_media': 'cobertura_bajo_riesgo_media'})
)


## 8. Ranking técnico transparente


In [ ]:
df_ranking = (
    df_resumen_modelos
    .merge(high_risk_coverage, on='politica', how='left')
    .merge(medium_risk_coverage, on='politica', how='left')
    .merge(low_risk_coverage, on='politica', how='left')
)

def minmax(series):
    minimum = series.min()
    maximum = series.max()
    if maximum == minimum:
        return pd.Series(1.0, index=series.index)
    return (series - minimum) / (maximum - minimum)

df_ranking['puntaje_recompensa'] = minmax(df_ranking['recompensa_media'])
df_ranking['puntaje_cobertura_alto'] = minmax(df_ranking['cobertura_alto_riesgo_media'])
df_ranking['puntaje_eventos'] = 1.0 - minmax(df_ranking['eventos_adversos_medios'])
df_ranking['puntaje_capacidad'] = 1.0 - minmax(df_ranking['excesos_capacidad_medios'])
df_ranking['puntaje_eficiencia_recursos'] = 1.0 - minmax(df_ranking['recursos_usados_medios'])

df_ranking['puntaje_tecnico'] = (
    0.35 * df_ranking['puntaje_recompensa']
    + 0.30 * df_ranking['puntaje_cobertura_alto']
    + 0.15 * df_ranking['puntaje_eventos']
    + 0.10 * df_ranking['puntaje_capacidad']
    + 0.10 * df_ranking['puntaje_eficiencia_recursos']
)

df_ranking['ranking_tecnico'] = (
    df_ranking['puntaje_tecnico']
    .rank(method='dense', ascending=False)
    .astype(int)
)

df_ranking = df_ranking.sort_values(['ranking_tecnico', 'politica']).reset_index(drop=True)
display(df_ranking[[
    'ranking_tecnico', 'politica', 'puntaje_tecnico', 'recompensa_media',
    'cobertura_alto_riesgo_media', 'eventos_adversos_medios',
    'recursos_usados_medios', 'excesos_capacidad_medios',
]].round(3))


## 9. Etiquetas de interpretación


In [ ]:
df_ranking['tiene_cobertura_alto_riesgo'] = df_ranking['cobertura_alto_riesgo_media'] > 0
df_ranking['respeta_capacidad_en_promedio'] = df_ranking['excesos_capacidad_medios'] == 0
df_ranking['tiene_contactos'] = df_ranking['contactos_medios'] > 0

def classify_policy(row):
    if not row['tiene_contactos']:
        return 'sin_contacto_efectivo'
    if not row['tiene_cobertura_alto_riesgo']:
        return 'sin_cobertura_alto_riesgo'
    if row['politica'] in {'aleatoria', 'reglas_riesgo', 'reglas_capacidad'}:
        return 'referencia_no_aprendida'
    return 'agente_aprendido_evaluable'

df_ranking['clasificacion'] = df_ranking.apply(classify_policy, axis=1)
display(df_ranking[[
    'ranking_tecnico', 'politica', 'clasificacion', 'tiene_contactos',
    'tiene_cobertura_alto_riesgo', 'respeta_capacidad_en_promedio',
]])


## 10. Guardar resultados


In [ ]:
CONSOLIDATED_PATH = TABLES_DIR / 'evaluacion_conjunta_modelos_v1.csv'
EPISODE_PATH = TABLES_DIR / 'evaluacion_episodios_modelos_v1.csv'
SUMMARY_PATH = TABLES_DIR / 'resumen_modelos_v1.csv'
RISK_PATH = TABLES_DIR / 'cobertura_por_riesgo_modelos_v1.csv'
RANKING_PATH = TABLES_DIR / 'ranking_tecnico_modelos_v1.csv'
PROTOCOL_PATH = TABLES_DIR / 'validacion_protocolo_modelos_v1.csv'

df_evaluacion_conjunta.to_csv(CONSOLIDATED_PATH, index=False, encoding='utf-8-sig')
episode_level.to_csv(EPISODE_PATH, index=False, encoding='utf-8-sig')
df_resumen_modelos.to_csv(SUMMARY_PATH, index=False, encoding='utf-8-sig')
df_cobertura_riesgo.to_csv(RISK_PATH, index=False, encoding='utf-8-sig')
df_ranking.to_csv(RANKING_PATH, index=False, encoding='utf-8-sig')
protocol_check.to_csv(PROTOCOL_PATH, index=False, encoding='utf-8-sig')

evaluation_metadata = {
    'version': '1.0',
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'seeds': [42, 123, 2026],
    'policies_expected': sorted(expected_policies),
    'steps_per_episode': expected_steps,
    'input_files': {name: str(path) for name, path in INPUT_FILES.items()},
    'ranking_weights': {
        'recompensa_media': 0.35,
        'cobertura_alto_riesgo': 0.30,
        'eventos_adversos': 0.15,
        'cumplimiento_capacidad': 0.10,
        'eficiencia_recursos': 0.10,
    },
    'scope_note': 'Evaluación sobre un simulador con transiciones y recompensas supuestas; no implica efectividad clínica real.',
}

metadata_path = METADATA_DIR / 'protocolo_evaluacion_modelos_v1.json'
with metadata_path.open('w', encoding='utf-8') as file:
    json.dump(evaluation_metadata, file, ensure_ascii=False, indent=2)

for path in [CONSOLIDATED_PATH, EPISODE_PATH, SUMMARY_PATH, RISK_PATH, RANKING_PATH, PROTOCOL_PATH, metadata_path]:
    print(path.name, '| existe:', path.is_file(), '| bytes:', path.stat().st_size if path.is_file() else 0)

print('Evaluación conjunta completada y guardada.')
